In [ ]:
# -*- coding: utf-8 -*-
# =====================================================================================
#  0번 셀 — 라이브러리 설치
#  새 Colab T4 런타임에서 가장 먼저 한 번 실행합니다.
#  torch는 Colab 기본 설치본을 그대로 사용하며 재설치하지 않습니다(재설치 금지 규정).
# =====================================================================================
import subprocess
import sys


def _install_baseline_packages():
    packages = [
        "fastapi",
        "uvicorn",
        "langgraph",
        "pydantic>=2",
        "sentence-transformers",
        "faiss-cpu",
        "transformers",
        "accelerate",
        "bitsandbytes",
        "pypdf",
        "python-docx",
        "beautifulsoup4",
        "requests",
    ]
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        check=True,
    )


_install_baseline_packages()
print("[0번 셀 완료] 라이브러리 설치 완료 — 다음 셀(결과기 로직)을 실행하세요.")


In [ ]:
# -*- coding: utf-8 -*-
"""
카카오 약관 RAG 결과기 — Baseline 구현본 (LangGraph)

스켈레톤의 모든 [STUB] 지점을 실제 로직으로 교체했다.
함수 시그니처와 스키마는 스켈레톤 원본을 그대로 유지하며, 각 함수가 필요로 하는
import는 모두 함수 본문 안에 둔다.

Baseline 구성
  로딩 requests + BeautifulSoup / 파싱 조 번호 정규식(순번 검증)
  청킹 조(article) 단위 / 임베딩 bge-m3 / 저장 FAISS IndexFlatIP
  검색 cosine top-k(k=4) / 생성 Qwen2.5-7B-Instruct 4bit
"""

from __future__ import annotations

import json
import threading
import time
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional, Tuple

from langgraph.graph import END, StateGraph
from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator

# ==================
# 0. 스텁 헬퍼
# ==================

STUB_LOG: List[str] = []


def stub(stage: str, detail: str = "") -> None:
    # detail이 있으면 " — detail"을 붙이고, 없으면 stage만 사용해 한 줄 메시지를 만든다
    line = f"[STUB] {stage}" + (f" — {detail}" if detail else "")
    # 만든 메시지를 전역 로그 리스트에 누적한다
    STUB_LOG.append(line)
    # 동시에 콘솔에도 즉시 출력한다
    print(line)


# ==================
# 1. 고정 상수
# ==================

DocName = Literal[
    "카카오계정 약관",
    "카카오 위치정보 이용약관",
    "카카오 통합서비스약관",
    "카카오 통합 약관",
]

OFFICIAL_DOCUMENT_NAMES: Tuple[str, ...] = (
    "카카오계정 약관",
    "카카오 위치정보 이용약관",
    "카카오 통합서비스약관",
    "카카오 통합 약관",
)

REQUIRED_GENERATION_MODEL_FAMILY = "Qwen2.5-Instruct"


def normalize_doc_name(value: Any) -> str:
    import re
    import unicodedata

    # 입력값을 문자열로 강제 변환한 뒤 NFC 정규화(자모 결합 형태 통일)를 적용하고,
    # 정규화된 문자열에서 공백 문자를 전부 제거해 반환한다
    return re.sub(r"\s+", "", unicodedata.normalize("NFC", str(value)))


# 공식 문서명 4종을 정규화한 집합으로 미리 만들어 둔다 (매번 재계산하지 않도록 캐시)
ALLOWED_DOCS_NORM = {normalize_doc_name(d) for d in OFFICIAL_DOCUMENT_NAMES}


# ==================
# 2. 스키마 — 설정
# ==================

class DocumentSource(BaseModel):
    doc_name: DocName
    urls: List[str] = Field(default_factory=list)
    local_path: Optional[str] = None
    effective_date: str
    note: str = ""

    @model_validator(mode="after")
    def _require_source(self) -> "DocumentSource":
        # urls도 없고 local_path도 없으면 원문을 가져올 방법이 전혀 없으므로 즉시 예외를 던진다
        if not self.urls and not self.local_path:
            raise ValueError(f"[{self.doc_name}] urls 또는 local_path 중 하나는 필요합니다.")
        # 검증을 통과하면 객체 자신을 그대로 반환한다 (pydantic model_validator 관례)
        return self


class IndexConfig(BaseModel):
    sources: List[DocumentSource]
    embedding_model_name: str = "BAAI/bge-m3"
    embedding_batch_size: int = 8
    max_chunk_chars: int = 1800
    request_timeout_s: float = 20.0


class RetrievalConfig(BaseModel):
    top_k: int = 4
    query_prefix: str = ""


class GenerationConfig(BaseModel):
    model_name: str = "Qwen/Qwen2.5-7B-Instruct"
    load_in_4bit: bool = True
    max_new_tokens: int = 512
    temperature: float = 0.0
    max_context_chars: int = 6000


class PipelineConfig(BaseModel):
    index: IndexConfig
    retrieval: RetrievalConfig = RetrievalConfig()
    generation: GenerationConfig = GenerationConfig()


# ==================
# 3. 스키마 — 인덱싱
# ==================

class RawDocument(BaseModel):
    doc_name: DocName
    text: str
    source_url: str
    fetched_at: str
    char_len: int


class Article(BaseModel):
    doc_name: DocName
    article_number: int
    article_title: str = ""
    body: str

    @property
    def citation(self) -> str:
        # "문서명 제N조" 형태의 머리글을 먼저 만든다
        head = f"{self.doc_name} 제{self.article_number}조"
        # 제목이 있으면 괄호를 붙이고, 없으면 머리글만 반환한다
        return f"{head}({self.article_title})" if self.article_title else head


class Chunk(BaseModel):
    chunk_id: str
    doc_name: DocName
    article_number: int
    article_title: str = ""
    text: str


class EmbeddingBundle(BaseModel):
    chunks: List[Chunk]
    vectors: List[List[float]]
    model_name: str
    dim: int


class IndexStats(BaseModel):
    n_documents: int
    n_articles: int
    n_chunks: int
    dim: int
    per_document: Dict[str, int]
    elapsed_s: float


# ==================
# 4. 스키마 — 검색 / 증강 / 생성
# ==================

class RetrievedChunk(BaseModel):
    rank: int
    score: float
    chunk: Chunk


class RetrievalOutput(BaseModel):
    question: str
    hits: List[RetrievedChunk]
    top_k: int
    elapsed_s: float


class PromptBundle(BaseModel):
    system_prompt: str
    user_prompt: str
    context_block: str
    n_context_chunks: int


class GenerationOutput(BaseModel):
    answer_text: str
    n_new_tokens: int
    elapsed_s: float


class Evidence(BaseModel):
    doc_name: DocName
    article_number: int

    def to_pair(self) -> List[Any]:
        # [문서명, 조번호] 2원소 리스트로 직렬화한다 (제출 형식과 동일)
        return [self.doc_name, int(self.article_number)]


class AnswerPayload(BaseModel):
    answer: str
    retrieved: List[Evidence] = Field(min_length=1, max_length=4)

    @field_validator("retrieved")
    @classmethod
    def _allowed_docs(cls, v: List[Evidence]) -> List[Evidence]:
        # retrieved 안의 문서명을 하나씩 검사한다
        for item in v:
            # 정규화한 문서명이 허용 목록에 없으면 즉시 예외를 던진다
            if normalize_doc_name(item.doc_name) not in ALLOWED_DOCS_NORM:
                raise ValueError(f"허용 목록 밖 문서명: {item.doc_name}")
        # 전부 통과하면 원래 리스트를 그대로 반환한다
        return v

    def to_contract(self) -> Dict[str, Any]:
        # answer는 그대로, retrieved는 각 Evidence를 [문서명, 조번호] 쌍으로 변환한 리스트로 감싼다
        return {"answer": self.answer, "retrieved": [e.to_pair() for e in self.retrieved]}


# ==================
# 5. 스키마 — 품질 결과 (골드셋 / 제출 파일)
# ==================

class GoldArticle(BaseModel):
    doc: DocName
    article: int
    citation: str


class GoldQuestion(BaseModel):
    id: str
    question: str
    ptype: str
    difficulty: str
    gold_articles: List[GoldArticle]
    key_facts: List[str]


class GoldSet(BaseModel):
    questions: List[GoldQuestion]
    meta: Dict[str, Any] = Field(default_factory=dict, alias="_meta")

    model_config = ConfigDict(populate_by_name=True)


class SubmissionAnswer(BaseModel):
    qid: str
    retrieved: List[List[Any]]
    answer: str
    error: Optional[str] = None


class SubmissionFile(BaseModel):
    team: str
    answers: List[SubmissionAnswer]
    meta: Dict[str, Any] = Field(default_factory=dict)


class ArticleScore(BaseModel):
    qid: str
    predicted: List[List[Any]]
    gold: List[List[Any]]
    hit_at_1: bool
    hit_at_k: bool
    n_gold_matched: int
    n_gold_total: int


class KeyFactScore(BaseModel):
    qid: str
    n_key_facts: int
    n_covered_auto: int
    coverage_auto: float
    per_fact: List[Dict[str, Any]]
    needs_manual_review: bool = True


class ItemReport(BaseModel):
    qid: str
    question: str
    difficulty: str
    ptype: str
    article: ArticleScore
    key_fact: KeyFactScore
    answer_text: str


class EvalReport(BaseModel):
    n_items: int
    article_hit_at_1_rate: float
    article_hit_at_k_rate: float
    key_fact_coverage_mean: float
    items: List[ItemReport]


class PerfProtocol(BaseModel):
    requests_per_run: int = 12
    concurrency: int = 2
    warmup_requests: int = 2
    repetitions: int = 3


class PerfReport(BaseModel):
    protocol: PerfProtocol
    success_rate: float
    throughput_rps: float
    p50_latency_s: Optional[float]
    p95_latency_s: Optional[float]


# ==================
# 6. 인덱싱 — 로딩 및 가져오기
# ==================

DEFAULT_SOURCES: List[DocumentSource] = [
    DocumentSource(
        doc_name="카카오계정 약관",
        urls=[
            "https://www.kakao.com/policy/terms?lang=ko",
            "https://qr.kakao.com/policy/terms?lang=ko",
            "https://t1.kakaocdn.net/kakaocorp/pw/policy/files/카카오계정약관.pdf",
        ],
        effective_date="2026-05-29",
        note="HTML 실패 시 PDF로 폴백. PDF 경로도 검색 시점 기준 추정값 — 실행 전 재확인 필요.",
    ),
    DocumentSource(
        doc_name="카카오 위치정보 이용약관",
        urls=[
            "https://www.kakao.com/policy/location?lang=ko",
            "https://qr.kakao.com/policy/location?lang=ko",
        ],
        effective_date="2026-07-16",
        note="HTML 경로만 확인됨. PDF 폴백 경로는 미확보.",
    ),
    DocumentSource(
        doc_name="카카오 통합서비스약관",
        urls=[
            "https://www.kakao.com/policy/terms?type=ts&lang=ko",
            "https://qr.kakao.com/policy/terms?type=ts&lang=ko",
        ],
        effective_date="2026-05-29",
        note="URL 실제 접근 가능 여부 확인 필요. PDF 폴백 경로는 미확보.",
    ),
    DocumentSource(
        doc_name="카카오 통합 약관",
        urls=[
            "https://www.kakao.com/policy/kakaoTerms?lang=ko",
        ],
        effective_date="2022-08-25",
        note="반드시 운영진 배포 공식 아카이브 링크(2022-08-25 시행본)로 교체할 것.",
    ),
]


def _load_local_file_text(local_path: str) -> Tuple[str, str]:
    from pathlib import Path

    # 노트북과 같은 위치, 또는 Colab 세션 루트(/content) 두 곳을 후보로 만든다
    candidates = [Path(local_path), Path("/content") / local_path]
    # 후보 중 실제로 존재하는 첫 번째 경로를 찾는다 (없으면 None)
    resolved = next((path for path in candidates if path.exists()), None)
    if resolved is None:
        # 아무 후보도 없으면 시도한 경로 목록을 메시지에 담아 예외를 던진다
        tried = ", ".join(str(path) for path in candidates)
        raise FileNotFoundError(f"로컬 파일을 찾을 수 없습니다. 시도한 경로: {tried}")

    # 확장자를 소문자로 통일해 분기 기준으로 사용한다
    suffix = resolved.suffix.lower()
    if suffix == ".pdf":
        from pypdf import PdfReader

        # PDF를 열어 페이지 객체 리스트를 얻는다
        reader = PdfReader(str(resolved))
        # 각 페이지의 텍스트를 추출하고(실패 시 빈 문자열), 줄바꿈으로 이어붙인다
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
    elif suffix == ".docx":
        from docx import Document

        # DOCX를 열어 문서 객체를 얻는다
        document = Document(str(resolved))
        # 빈 문단은 제외하고 문단 텍스트만 줄바꿈으로 이어붙인다
        text = "\n".join(p.text for p in document.paragraphs if p.text.strip())
    else:
        # pdf, docx가 아닌 확장자는 처리 방법이 없으므로 예외를 던진다
        raise ValueError(f"지원하지 않는 로컬 파일 형식: {suffix} ({resolved})")

    # 추출한 텍스트와 실제로 읽은 경로를 튜플로 반환한다
    return text, str(resolved)


def fetch_document(source: DocumentSource, timeout_s: float = 20.0) -> RawDocument:
    import datetime
    import re

    # "제N조" 패턴 검출용 정규식 (조 구조가 있는지 판별하는 데 사용)
    article_pattern = re.compile(r"제\s*\d+\s*조")

    def finalize(raw_text: str) -> str:
        # 줄바꿈 없는 공백(&nbsp; 등)을 일반 스페이스로 치환한다
        cleaned = raw_text.replace("\u00a0", " ")
        # 줄 단위로 쪼개 각 줄의 앞뒤 공백을 제거한다
        lines = [line.strip() for line in cleaned.split("\n")]
        # 빈 줄은 제거하고 나머지를 다시 줄바꿈으로 이어붙인다
        return "\n".join(line for line in lines if line)

    if source.local_path:
        # 로컬 경로가 지정된 경우, 로컬 파일에서 원문과 실제 경로를 읽어온다
        raw_text, resolved_path = _load_local_file_text(source.local_path)
        # 읽어온 원문을 정리한다
        text = finalize(raw_text)
        if len(article_pattern.findall(text)) < 3:
            # 조 패턴이 3개 미만이면 이 문서는 조 구조를 갖추지 못한 것으로 보고 예외를 던진다
            raise RuntimeError(
                f"[{source.doc_name}] 조 구조 미검출 · 경로={resolved_path} · len={len(text)}"
            )
        # 정상이면 RawDocument로 감싸 즉시 반환한다 (URL 폴백 로직은 실행되지 않음)
        return RawDocument(
            doc_name=source.doc_name,
            text=text,
            source_url=resolved_path,
            fetched_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),
            char_len=len(text),
        )

    import requests
    from bs4 import BeautifulSoup

    # 일반 브라우저처럼 보이도록 User-Agent와 언어 헤더를 지정한다
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
        ),
        "Accept-Language": "ko-KR,ko;q=0.9",
    }
    # URL 후보별 실패 사유를 모아두는 리스트 (전부 실패했을 때 메시지로 사용)
    failures: List[str] = []

    def is_pdf_response(url: str, response: "requests.Response") -> bool:
        # 응답 헤더의 Content-Type을 소문자로 확인한다
        content_type = response.headers.get("Content-Type", "").lower()
        # Content-Type이 PDF이거나 URL 확장자가 .pdf면 PDF 응답으로 판단한다
        return "application/pdf" in content_type or url.lower().endswith(".pdf")

    # urls 리스트를 순서대로 하나씩 시도한다
    for url in source.urls:
        try:
            # GET 요청을 보낸다
            response = requests.get(url, headers=headers, timeout=timeout_s)
            # 4xx/5xx 응답이면 여기서 예외를 발생시켜 except로 넘어간다
            response.raise_for_status()

            if is_pdf_response(url, response):
                from io import BytesIO

                from pypdf import PdfReader

                # 응답 바이트를 메모리 파일처럼 다룰 수 있게 감싼 뒤 PDF로 연다
                reader = PdfReader(BytesIO(response.content))
                # 각 페이지 텍스트를 추출해 이어붙인다
                raw_text = "\n".join(page.extract_text() or "" for page in reader.pages)
            else:
                # 인코딩을 응답이 추정한 값(없으면 utf-8)으로 설정한다
                response.encoding = response.apparent_encoding or "utf-8"
                # HTML을 파서로 로드한다
                soup = BeautifulSoup(response.text, "html.parser")
                # 본문과 무관한 태그(script, style 등)를 통째로 제거한다
                for tag in soup(["script", "style", "noscript", "header", "footer", "nav"]):
                    tag.decompose()

                # 실제 약관 본문 컨테이너만 선택한다 (목차는 이 요소 바깥의 형제 요소라 자동 배제됨)
                content = soup.select_one("div.wrap_terms.wrap_policy")
                if content is None:
                    # 이 선택자가 없는 템플릿이면 경고를 남기고 페이지 전체로 폴백한다
                    print(
                        f"[경고][로딩] {source.doc_name} — "
                        "'div.wrap_terms.wrap_policy' 컨테이너를 찾지 못해 "
                        "전체 페이지에서 추출합니다(목차 혼입 가능, span 휴리스틱에 의존)."
                    )
                    content = soup

                # 선택된 요소의 텍스트만 줄바꿈으로 구분해 추출한다
                raw_text = content.get_text(separator="\n")

            # 추출한 원문을 정리한다
            text = finalize(raw_text)

            if len(article_pattern.findall(text)) < 3:
                # 조 패턴이 3개 미만이면 이 URL은 실패로 기록하고 다음 URL을 시도한다
                failures.append(f"{url}: 조 구조 미검출(len={len(text)})")
                continue

            # 조 구조가 확인되면 RawDocument로 감싸 즉시 반환한다 (남은 URL은 시도하지 않음)
            return RawDocument(
                doc_name=source.doc_name,
                text=text,
                source_url=url,
                fetched_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),
                char_len=len(text),
            )
        except Exception as exc:
            # 요청/파싱 중 어떤 예외가 나든 실패 목록에 기록하고 다음 URL로 넘어간다
            failures.append(f"{url}: {type(exc).__name__}: {exc}")

    # 모든 URL을 다 시도했는데도 성공하지 못하면 실패 사유를 모아 예외를 던진다
    raise RuntimeError(f"[{source.doc_name}] 원문 취득 실패 — " + " | ".join(failures))


def load_documents(
    sources: List[DocumentSource],
    timeout_s: float = 20.0,
) -> List[RawDocument]:
    # 결과를 담을 빈 리스트를 만든다
    documents: List[RawDocument] = []
    # 소스를 하나씩 순회한다
    for source in sources:
        # 개별 소스에서 문서를 가져온다
        document = fetch_document(source, timeout_s)
        # 어떤 문서를 어디서 몇 자 가져왔는지 로그로 남긴다
        print(
            f"[로딩] {document.doc_name} · {document.char_len}자 · {document.source_url}"
        )
        # 결과 리스트에 추가한다
        documents.append(document)
    # 전체 문서 리스트를 반환한다
    return documents


# ==================
# 7. 인덱싱 — 파싱
# ==================

def parse_articles(document: RawDocument) -> List[Article]:
    import re

    text = document.text
    # "제N조" 패턴에서 숫자를 캡처 그룹으로 뽑는 정규식
    header_pattern = re.compile(r"제\s*(\d+)\s*조")
    # "제N조" 뒤에 바로 이 접미사가 오면 본문 중 상호참조("제5조에 따른" 등)로 간주해 제외
    reference_suffix = ("에", "의", "와", "과", "및", "부터", "까지", "에서", ",", "제")

    # (시작 위치, 헤더 끝 위치, 조번호, 제목) 튜플을 모으는 리스트
    candidates: List[Tuple[int, int, int, str]] = []

    # 텍스트 전체에서 "제N조" 패턴을 순서대로 찾는다
    for match in header_pattern.finditer(text):
        # 매치된 문자열에서 조 번호를 정수로 변환한다
        number = int(match.group(1))

        # 매치 뒤 90자를 미리 잘라 제목/참조 여부 판단에 사용한다
        tail = text[match.end():match.end() + 90]
        # 앞쪽 공백을 제거한 버전을 만든다
        stripped_tail = tail.lstrip()
        if stripped_tail and stripped_tail[0] in reference_suffix:
            # 첫 글자가 참조 접미사이면 상호참조로 보고 이 매치는 건너뛴다
            continue

        # "(제목)" 형태를 먼저 시도한다
        title_match = re.match(r"[ \t]*\(([^)\n]{1,60})\)", tail)
        if title_match:
            # 괄호 안 문자열을 제목으로 쓰고, 그 뒤까지를 헤더 끝 위치로 잡는다
            title = title_match.group(1).strip()
            header_end = match.end() + title_match.end()
        else:
            # 괄호가 없으면 다음 줄바꿈 전까지(최대 60자)를 후보 제목으로 본다
            line_match = re.match(r"[ \t]*([^\n]{0,60})", tail)
            candidate = line_match.group(1).strip() if line_match else ""
            # 혹시 줄바꿈이 포함됐으면 첫 줄만 남긴다
            candidate = candidate.split("\n")[0].strip()
            # 길이가 1~40자 범위일 때만 제목으로 채택한다
            title = candidate if 0 < len(candidate) <= 40 else ""
            # 제목을 채택했으면 그 길이만큼 헤더 끝 위치를 밀고, 아니면 매치 끝 그대로 둔다
            header_end = match.end() + (len(line_match.group(0)) if title else 0)

        # 이번 매치를 후보 목록에 추가한다
        candidates.append((match.start(), header_end, number, title))

    # 1부터 연속 증가하는 구간(run)들을 담을 리스트
    runs: List[List[Tuple[int, int, int, str]]] = []
    # 현재 만들고 있는 구간
    current_run: List[Tuple[int, int, int, str]] = []
    # 다음에 나와야 할 조 번호
    expected_number = 1

    # 후보를 순서대로 확인하며 구간을 만든다
    for candidate in candidates:
        _, _, number, _ = candidate
        if number == expected_number:
            # 기대한 번호와 일치하면 현재 구간에 추가하고 기대값을 1 올린다
            current_run.append(candidate)
            expected_number += 1
        elif number == 1:
            # 기대값은 아니지만 번호가 1이면 새 구간이 시작된 것으로 보고,
            # 지금까지 만든 구간을 저장한 뒤 새 구간을 1부터 다시 연다
            if current_run:
                runs.append(current_run)
            current_run = [candidate]
            expected_number = 2
        # 기대값도 아니고 1도 아니면 잡음으로 보고 그냥 무시한다 (else 없음)

    if current_run:
        # 마지막으로 만들던 구간이 남아 있으면 저장한다
        runs.append(current_run)

    if not runs:
        # 구간이 하나도 없으면 조 구조를 전혀 못 찾은 것이므로 예외를 던진다
        raise ValueError(f"[{document.doc_name}] 조 구조 파싱 실패 — 정규식 재검토 필요")

    def run_span(run: List[Tuple[int, int, int, str]]) -> int:
        # 구간의 마지막 헤더 시작 위치에서 첫 헤더 시작 위치를 빼 폭(글자 수)을 구한다
        return run[-1][0] - run[0][0]

    # 폭이 가장 큰 구간을 실제 본문으로 채택한다 (목차는 폭이 좁음)
    headers = max(runs, key=run_span)

    if len(runs) > 1:
        # 구간이 여러 개였다면(목차 중복 가능성) 각 구간의 크기를 로그로 남긴다
        detail = ", ".join(f"{len(run)}개조/span={run_span(run)}자" for run in runs)
        print(
            f"[경고][파싱] {document.doc_name} 조 시퀀스 {len(runs)}개 발견"
            f"(목차 등 중복 가능성) — {detail} · 가장 긴 구간을 본문으로 채택"
        )

    # "제N장 ..." 형태의 장 제목 줄을 제거하기 위한 정규식
    chapter_pattern = re.compile(r"^제\s*\d+\s*장.*$", re.MULTILINE)
    articles: List[Article] = []

    # 채택된 헤더들을 순서대로 순회하며 각 조의 본문 구간을 잘라낸다
    for index, (_, header_end, number, title) in enumerate(headers):
        # 다음 헤더 시작 위치까지, 마지막 조라면 텍스트 끝까지를 본문 끝으로 삼는다
        body_end = headers[index + 1][0] if index + 1 < len(headers) else len(text)
        # 헤더 끝부터 본문 끝까지를 잘라낸다
        body = text[header_end:body_end]
        # 본문 안에 섞여 있을 수 있는 장 제목 줄을 제거한다
        body = chapter_pattern.sub("", body)
        # 연속된 줄바꿈을 하나로 줄이고 앞뒤 공백을 제거한다
        body = re.sub(r"\n{2,}", "\n", body).strip()

        # 정리된 조 정보를 Article 객체로 만들어 리스트에 추가한다
        articles.append(
            Article(
                doc_name=document.doc_name,
                article_number=number,
                article_title=title,
                body=body,
            )
        )

    if not articles:
        # 헤더는 있었지만 결과 리스트가 비었다면(이론상 발생하지 않아야 함) 예외를 던진다
        raise ValueError(f"[{document.doc_name}] 조 구조 파싱 실패 — 정규식 재검토 필요")

    # 본문이 20자 미만인 조만 골라 "제N조(len=..)" 형태 문자열 리스트로 만든다
    suspicious = [
        f"제{a.article_number}조(len={len(a.body)})"
        for a in articles
        if len(a.body) < 20
    ]
    if suspicious:
        # 의심스러운 조가 있으면 목록을 경고로 출력한다
        print(
            f"[경고][파싱] {document.doc_name} 본문이 20자 미만인 조 {len(suspicious)}건 "
            f"— 제목 추출 로직이 본문을 흡수했을 가능성: " + ", ".join(suspicious)
        )

    # 최종적으로 몇 조부터 몇 조까지 몇 개를 파싱했는지 로그로 남긴다
    print(f"[파싱] {document.doc_name} · 제1조~제{articles[-1].article_number}조 "
          f"({len(articles)}개)")
    # 완성된 조 리스트를 반환한다
    return articles


# ==================
# 8. 인덱싱 — 청킹
# ==================

def chunk_articles(articles: List[Article], config: IndexConfig) -> List[Chunk]:
    # 결과 청크를 담을 리스트
    chunks: List[Chunk] = []
    # 본문이 비어 있던 조 번호를 기록할 리스트 (경고 출력용)
    empty_body_articles: List[str] = []

    # 조를 하나씩 순회한다
    for article in articles:
        # "[문서명] 제N조" 형태의 머리글을 만든다
        header_line = f"[{article.doc_name}] 제{article.article_number}조"
        if article.article_title:
            # 제목이 있으면 괄호로 덧붙인다
            header_line += f"({article.article_title})"

        body = article.body
        if not body:
            # 본문이 비어 있으면 실패 목록에 번호를 기록하고,
            empty_body_articles.append(f"제{article.article_number}조")
            # 대신 "본문 파싱 실패" 표시가 담긴 청크 하나를 만들어 인덱스 손실을 막는다
            chunks.append(
                Chunk(
                    chunk_id=(
                        f"{normalize_doc_name(article.doc_name)}"
                        f"-{article.article_number:03d}-00"
                    ),
                    doc_name=article.doc_name,
                    article_number=article.article_number,
                    article_title=article.article_title,
                    text=f"{header_line}\n(본문 파싱 실패 — 원문 확인 필요)",
                )
            )
            # 이 조는 여기서 처리를 끝내고 다음 조로 넘어간다
            continue

        # 청크 하나에 담을 수 있는 최대 글자 수를 계산한다 (머리글 길이만큼 빼고, 최소 200자 보장)
        budget = max(200, config.max_chunk_chars - len(header_line) - 1)

        if len(body) <= budget:
            # 본문이 예산 안에 들어오면 통째로 청크 하나로 만든다
            segments = [body]
        else:
            # 예산을 넘으면 여러 조각으로 나눈다
            segments = []
            cursor = 0
            while cursor < len(body):
                # 이번 조각의 끝 위치를 예산 기준으로 잠정 결정한다
                window_end = min(cursor + budget, len(body))
                if window_end < len(body):
                    # 본문 끝이 아니라면, 자연스러운 경계(줄바꿈)를 찾아 자른다
                    boundary = body.rfind("\n", cursor + budget // 2, window_end)
                    if boundary == -1:
                        # 줄바꿈이 없으면 마침표+공백 경계를 찾는다
                        boundary = body.rfind(". ", cursor + budget // 2, window_end)
                    if boundary != -1:
                        # 경계를 찾았으면 그 위치(포함) 다음까지로 자른다
                        window_end = boundary + 1
                # 이번 조각을 잘라내 앞뒤 공백을 제거하고 리스트에 추가한다
                segments.append(body[cursor:window_end].strip())
                # 다음 조각은 이번 조각이 끝난 위치부터 시작한다
                cursor = window_end

        # 혹시 빈 조각이 섞였으면 제거하고, 전부 비었으면 빈 문자열 하나로 대체한다
        segments = [s for s in segments if s] or [""]

        # 조각마다 청크 객체를 만들어 결과 리스트에 추가한다
        for part_index, segment in enumerate(segments):
            chunks.append(
                Chunk(
                    chunk_id=(
                        f"{normalize_doc_name(article.doc_name)}"
                        f"-{article.article_number:03d}-{part_index:02d}"
                    ),
                    doc_name=article.doc_name,
                    article_number=article.article_number,
                    article_title=article.article_title,
                    text=f"{header_line}\n{segment}",
                )
            )

    if empty_body_articles:
        # 본문이 비었던 조가 있었다면 어느 조였는지 경고로 출력한다
        print(
            f"[경고][청킹] {articles[0].doc_name} 본문 파싱 실패 {len(empty_body_articles)}건: "
            + ", ".join(empty_body_articles)
        )

    # 조 개수와 최종 청크 개수를 로그로 남긴다
    print(f"[청킹] {len(articles)}개 조항 → {len(chunks)}개 청크")
    # 완성된 청크 리스트를 반환한다
    return chunks


# ==================
# 9. 인덱싱 — 임베딩
# ==================

class EmbedderHandle(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    model_name: str
    dim: int
    device: str = "cpu"
    model: Any = None


def build_embedder(config: IndexConfig) -> EmbedderHandle:
    import torch
    from sentence_transformers import SentenceTransformer

    # CUDA GPU를 쓸 수 있으면 cuda, 없으면 cpu를 device로 선택한다
    device = "cuda" if torch.cuda.is_available() else "cpu"
    # 지정된 임베딩 모델을 해당 device에 로드한다
    model = SentenceTransformer(config.embedding_model_name, device=device)
    # 모델이 만드는 임베딩 벡터의 차원 수를 정수로 읽어온다
    dim = int(model.get_sentence_embedding_dimension())

    # 어떤 모델을 어느 device에 몇 차원으로 로드했는지 로그로 남긴다
    print(f"[임베딩 모델] {config.embedding_model_name} · device={device} · dim={dim}")
    # 로드한 모델과 메타정보를 핸들 객체로 감싸 반환한다
    return EmbedderHandle(
        model_name=config.embedding_model_name, dim=dim, device=device, model=model
    )


def embed_chunks(
    chunks: List[Chunk],
    embedder: EmbedderHandle,
    config: IndexConfig,
) -> EmbeddingBundle:
    # 청크 리스트에서 임베딩 대상 텍스트만 뽑아낸다
    texts = [chunk.text for chunk in chunks]
    # 배치 단위로 인코딩하고, 벡터를 L2 정규화하며(내적=cosine이 되도록), numpy 배열로 받는다
    vectors = embedder.model.encode(
        texts,
        batch_size=config.embedding_batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )

    # 몇 개 청크를 임베딩했고 결과 배열의 shape이 어떤지 로그로 남긴다
    print(f"[임베딩] {len(chunks)}개 청크 · shape={tuple(vectors.shape)}")
    # 청크 리스트와 벡터(파이썬 리스트로 변환), 모델명, 차원을 묶어 반환한다
    return EmbeddingBundle(
        chunks=chunks,
        vectors=vectors.astype("float32").tolist(),
        model_name=embedder.model_name,
        dim=int(vectors.shape[1]),
    )


# ==================
# 10. 인덱싱 — 저장
# ==================

class VectorStoreHandle(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    backend: str = "faiss.IndexFlatIP"
    dim: int
    n_vectors: int
    chunks: List[Chunk]
    index: Any = None

    def search(self, query_vector: List[float], top_k: int) -> List[RetrievedChunk]:
        import numpy as np

        # 질의 벡터를 FAISS가 요구하는 2차원 float32 배열로 변환한다
        query = np.asarray([query_vector], dtype="float32")
        # 저장된 벡터 개수와 top_k 중 작은 값만큼 검색해 점수와 인덱스를 얻는다
        scores, indices = self.index.search(query, min(top_k, self.n_vectors))

        # 결과를 담을 리스트
        hits: List[RetrievedChunk] = []
        # 점수와 위치를 순서쌍으로 묶어 순위(1부터)와 함께 순회한다
        for rank, (score, position) in enumerate(zip(scores[0], indices[0]), start=1):
            if position < 0:
                # FAISS가 유효한 결과를 못 찾으면 -1을 반환하므로 건너뛴다
                continue
            # 순위, 점수, 해당 위치의 청크를 묶어 결과에 추가한다
            hits.append(
                RetrievedChunk(
                    rank=rank,
                    score=round(float(score), 4),
                    chunk=self.chunks[int(position)],
                )
            )
        # 검색된 청크 리스트를 반환한다
        return hits


def build_vector_store(bundle: EmbeddingBundle) -> VectorStoreHandle:
    import faiss
    import numpy as np

    # 임베딩 리스트를 float32 numpy 배열로 변환한다
    vectors = np.asarray(bundle.vectors, dtype="float32")
    # 내적 기반(코사인 유사도용) 평면 인덱스를 생성한다
    index = faiss.IndexFlatIP(bundle.dim)
    # 벡터 전체를 인덱스에 추가한다
    index.add(vectors)

    # 몇 개 벡터를 몇 차원으로 저장했는지 로그로 남긴다
    print(f"[저장] FAISS IndexFlatIP · {index.ntotal}개 벡터 · dim={bundle.dim}")
    # 인덱스와 청크 리스트, 메타정보를 핸들 객체로 감싸 반환한다
    return VectorStoreHandle(
        dim=bundle.dim,
        n_vectors=int(index.ntotal),
        chunks=bundle.chunks,
        index=index,
    )


def build_index(config: IndexConfig) -> Tuple[VectorStoreHandle, EmbedderHandle, IndexStats]:
    # 전체 인덱싱 소요 시간을 재기 위해 시작 시각을 기록한다
    started = time.perf_counter()
    # 설정된 소스들에서 원문 문서를 전부 가져온다
    documents = load_documents(config.sources, config.request_timeout_s)

    # 전체 조를 모을 리스트
    articles: List[Article] = []
    # 문서마다 조 단위로 파싱해 리스트에 이어붙인다
    for document in documents:
        articles.extend(parse_articles(document))

    # 조 리스트를 청크로 분할한다
    chunks = chunk_articles(articles, config)
    # 임베딩 모델을 로드한다
    embedder = build_embedder(config)
    # 청크를 임베딩한다
    bundle = embed_chunks(chunks, embedder, config)
    # 임베딩 결과로 벡터 저장소를 만든다
    store = build_vector_store(bundle)

    # 문서별 청크 개수를 세기 위한 딕셔너리
    per_document: Dict[str, int] = {}
    for chunk in chunks:
        # 해당 문서명의 카운트를 1 증가시킨다 (없으면 0에서 시작)
        per_document[chunk.doc_name] = per_document.get(chunk.doc_name, 0) + 1

    # 인덱싱 통계를 하나의 객체로 정리한다
    stats = IndexStats(
        n_documents=len(documents),
        n_articles=len(articles),
        n_chunks=len(chunks),
        dim=store.dim,
        per_document=per_document,
        elapsed_s=round(time.perf_counter() - started, 4),
    )
    # 벡터 저장소, 임베더, 통계를 튜플로 반환한다
    return store, embedder, stats


# ==================
# 11. 검색
# ==================

def embed_query(
    question: str,
    embedder: EmbedderHandle,
    config: RetrievalConfig,
) -> List[float]:
    # 질문 앞에 접두어(있다면)를 붙여 인코딩하고, 정규화된 벡터의 첫 번째(유일한) 결과를 꺼낸다
    vector = embedder.model.encode(
        [config.query_prefix + question],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )[0]
    # float32 파이썬 리스트로 변환해 반환한다
    return vector.astype("float32").tolist()


def retrieve(
    question: str,
    store: VectorStoreHandle,
    embedder: EmbedderHandle,
    config: RetrievalConfig,
) -> RetrievalOutput:
    # 검색 소요 시간 측정을 위해 시작 시각을 기록한다
    started = time.perf_counter()
    # 질문을 벡터로 변환한다
    query_vector = embed_query(question, embedder, config)
    # 벡터 저장소에서 상위 top_k개를 검색한다
    hits = store.search(query_vector, config.top_k)
    # 질문, 검색 결과, top_k, 걸린 시간을 묶어 반환한다
    return RetrievalOutput(
        question=question,
        hits=hits,
        top_k=config.top_k,
        elapsed_s=round(time.perf_counter() - started, 4),
    )


# ==================
# 12. 증강
# ==================

SYSTEM_PROMPT = (
    "당신은 카카오 약관 전문 어시스턴트입니다. 제공된 약관 조문만을 근거로 답하십시오. "
    "근거에 없는 내용은 지어내지 말고, 숫자·기간·조문 번호는 근거 그대로 옮기십시오."
)


def build_prompt(retrieval: RetrievalOutput, config: GenerationConfig) -> PromptBundle:
    # 프롬프트에 넣을 근거 블록들을 모을 리스트
    blocks: List[str] = []
    # 지금까지 사용한 글자 수 누적값
    used_chars = 0

    # 검색된 청크를 순위대로 순회한다
    for hit in retrieval.hits:
        # "[근거 N] 문서명 / 조번호 / 본문" 형태의 블록 문자열을 만든다
        block = (
            f"[근거 {hit.rank}] 문서명: {hit.chunk.doc_name} / "
            f"조번호: 제{hit.chunk.article_number}조\n"
            f"본문: {hit.chunk.text}"
        )
        if blocks and used_chars + len(block) > config.max_context_chars:
            # 이미 근거가 하나 이상 있고, 이 블록을 더하면 예산을 넘기면 여기서 멈춘다
            break
        if not blocks and len(block) > config.max_context_chars:
            # 첫 블록부터 예산을 넘기면(근거가 하나도 없는 상태를 막기 위해) 잘라서라도 넣는다
            block = block[: config.max_context_chars]
        # 블록을 리스트에 추가한다
        blocks.append(block)
        # 누적 글자 수를 갱신한다
        used_chars += len(block)

    # 블록들을 빈 줄로 이어붙여 하나의 컨텍스트 문자열로 만든다
    context_block = "\n\n".join(blocks)
    # 컨텍스트, 질문, 답변 지시문을 합쳐 사용자 프롬프트를 완성한다
    user_prompt = (
        f"{context_block}\n\n"
        f"질문: {retrieval.question}\n\n"
        "위 근거 조문만을 사용해 한국어로 답하십시오. "
        "답변 안에 근거가 된 조문 번호를 함께 밝히고, "
        "근거에서 확인되지 않는 내용은 '제공된 약관 조문에서 확인할 수 없습니다'라고 답하십시오."
    )

    # 시스템 프롬프트, 사용자 프롬프트, 컨텍스트, 사용한 블록 수를 묶어 반환한다
    return PromptBundle(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=user_prompt,
        context_block=context_block,
        n_context_chunks=len(blocks),
    )


# ==================
# 13. 생성
# ==================

class GeneratorHandle(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    model_name: str
    load_in_4bit: bool
    model: Any = None
    tokenizer: Any = None


def load_generator(config: GenerationConfig) -> GeneratorHandle:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    if REQUIRED_GENERATION_MODEL_FAMILY.split("-")[0] not in config.model_name:
        # 모델명에 필수 계열 이름이 없으면 규정 위반이므로 예외를 던진다
        raise ValueError(f"생성 모델은 {REQUIRED_GENERATION_MODEL_FAMILY} 계열이어야 합니다.")

    # 지정된 모델명으로 토크나이저를 로드한다
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)

    # 모델 로드 시 공통으로 쓸 인자 딕셔너리 (device_map="auto"로 GPU/CPU 자동 배치)
    load_kwargs: Dict[str, Any] = {"device_map": "auto"}
    if config.load_in_4bit and torch.cuda.is_available():
        # 4bit 로드가 켜져 있고 GPU가 있으면 양자화 설정을 추가한다
        load_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
    else:
        # 그 외의 경우 float16 정밀도로 로드하도록 지정한다
        load_kwargs["torch_dtype"] = torch.float16

    # 위에서 구성한 인자로 실제 모델을 로드한다
    model = AutoModelForCausalLM.from_pretrained(config.model_name, **load_kwargs)
    # 추론 전용 모드(dropout 등 비활성화)로 전환한다
    model.eval()

    # 어떤 모델을 4bit 여부와 함께 로드했는지 로그로 남긴다
    print(f"[생성 모델] {config.model_name} · 4bit={config.load_in_4bit}")
    # 모델, 토크나이저, 메타정보를 핸들 객체로 감싸 반환한다
    return GeneratorHandle(
        model_name=config.model_name,
        load_in_4bit=config.load_in_4bit,
        model=model,
        tokenizer=tokenizer,
    )


def generate(prompt: PromptBundle, generator: GeneratorHandle) -> GenerationOutput:
    import torch

    # 생성 소요 시간 측정을 위해 시작 시각을 기록한다
    started = time.perf_counter()
    tokenizer = generator.tokenizer

    # 시스템/사용자 프롬프트를 채팅 메시지 형식으로 구성한다
    messages = [
        {"role": "system", "content": prompt.system_prompt},
        {"role": "user", "content": prompt.user_prompt},
    ]
    # 모델 전용 채팅 템플릿을 적용해 하나의 문자열로 만든다 (생성 시작 토큰 포함)
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    # 문자열을 토큰화하고 모델이 있는 device로 옮긴다
    model_inputs = tokenizer([text], return_tensors="pt").to(generator.model.device)

    # 그래디언트 계산 없이(추론 전용) 텍스트를 생성한다
    with torch.inference_mode():
        generated = generator.model.generate(
            **model_inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )

    # 입력 프롬프트의 토큰 길이를 구한다
    input_length = model_inputs["input_ids"].shape[1]
    # 생성 결과에서 입력 길이 이후, 즉 새로 생성된 토큰만 잘라낸다
    new_token_ids = generated[0][input_length:]
    # 새 토큰만 텍스트로 디코딩하고(특수 토큰 제거) 앞뒤 공백을 정리한다
    answer_text = tokenizer.decode(new_token_ids, skip_special_tokens=True).strip()

    # 답변 텍스트, 새로 생성된 토큰 수, 걸린 시간을 묶어 반환한다
    return GenerationOutput(
        answer_text=answer_text,
        n_new_tokens=int(new_token_ids.shape[0]),
        elapsed_s=round(time.perf_counter() - started, 4),
    )


def select_evidence(retrieval: RetrievalOutput, max_items: int = 4) -> List[Evidence]:
    # 최종 근거 리스트
    evidence: List[Evidence] = []
    # 이미 담은 (문서명, 조번호) 쌍을 기록해 중복을 막는 집합
    seen: set[Tuple[str, int]] = set()
    # 검색 결과를 순위대로 순회한다
    for hit in retrieval.hits:
        # 이 청크의 (문서명, 조번호) 키를 만든다
        key = (hit.chunk.doc_name, hit.chunk.article_number)
        if key in seen:
            # 이미 같은 조를 근거로 담았으면 건너뛴다 (같은 조가 여러 청크로 쪼개졌을 수 있으므로)
            continue
        # 처음 보는 키면 집합에 기록한다
        seen.add(key)
        # Evidence 객체로 만들어 리스트에 추가한다
        evidence.append(
            Evidence(doc_name=hit.chunk.doc_name, article_number=hit.chunk.article_number)
        )
        if len(evidence) >= max_items:
            # 최대 개수에 도달하면 더 담지 않고 멈춘다
            break
    if not evidence:
        # 검색 결과가 아예 없었다면(이론상 드묾) 최소 1개 요구사항을 맞추기 위해 기본값을 넣는다
        evidence.append(Evidence(doc_name="카카오 통합 약관", article_number=1))
    # 완성된 근거 리스트를 반환한다
    return evidence


# ==================
# 14. LangGraph 오케스트레이션
# ==================

class RagState(BaseModel):
    question: str
    retrieval: Optional[RetrievalOutput] = None
    prompt: Optional[PromptBundle] = None
    generation: Optional[GenerationOutput] = None
    payload: Optional[AnswerPayload] = None


class PipelineContext(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    store: VectorStoreHandle
    embedder: EmbedderHandle
    generator: GeneratorHandle
    config: PipelineConfig


def make_retrieve_node(ctx: PipelineContext):
    def retrieve_node(state: RagState) -> Dict[str, Any]:
        # 현재 질문으로 검색을 수행하고, 결과를 상태의 retrieval 필드 갱신값으로 반환한다
        return {"retrieval": retrieve(state.question, ctx.store, ctx.embedder, ctx.config.retrieval)}

    # 그래프에 등록할 노드 함수를 반환한다
    return retrieve_node


def make_augment_node(ctx: PipelineContext):
    def augment_node(state: RagState) -> Dict[str, Any]:
        # 이전 단계(retrieve)가 반드시 실행됐어야 한다는 전제를 검사한다
        assert state.retrieval is not None
        # 검색 결과로 프롬프트를 구성하고, 상태의 prompt 필드 갱신값으로 반환한다
        return {"prompt": build_prompt(state.retrieval, ctx.config.generation)}

    return augment_node


def make_generate_node(ctx: PipelineContext):
    def generate_node(state: RagState) -> Dict[str, Any]:
        # 이전 단계(augment)가 반드시 실행됐어야 한다는 전제를 검사한다
        assert state.prompt is not None
        # 프롬프트로 텍스트를 생성하고, 상태의 generation 필드 갱신값으로 반환한다
        return {"generation": generate(state.prompt, ctx.generator)}

    return generate_node


def make_finalize_node(ctx: PipelineContext):
    def finalize_node(state: RagState) -> Dict[str, Any]:
        # 검색과 생성이 모두 끝났다는 전제를 검사한다
        assert state.retrieval is not None and state.generation is not None
        # 생성된 답변(비어 있으면 기본 문구)과, 검색 결과에서 뽑은 근거로 최종 페이로드를 만든다
        payload = AnswerPayload(
            answer=state.generation.answer_text or "제공된 약관 조문에서 확인할 수 없습니다.",
            retrieved=select_evidence(state.retrieval, ctx.config.retrieval.top_k),
        )
        # 상태의 payload 필드 갱신값으로 반환한다
        return {"payload": payload}

    return finalize_node


def build_graph(ctx: PipelineContext):
    # RagState를 상태 타입으로 하는 그래프를 만든다
    graph = StateGraph(RagState)
    # 4개 노드를 각각 이름과 함께 등록한다
    graph.add_node("retrieve", make_retrieve_node(ctx))
    graph.add_node("augment", make_augment_node(ctx))
    graph.add_node("generate", make_generate_node(ctx))
    graph.add_node("finalize", make_finalize_node(ctx))

    # 시작 노드를 retrieve로 지정한다
    graph.set_entry_point("retrieve")
    # retrieve → augment → generate → finalize → 종료 순서로 엣지를 연결한다
    graph.add_edge("retrieve", "augment")
    graph.add_edge("augment", "generate")
    graph.add_edge("generate", "finalize")
    graph.add_edge("finalize", END)
    # 그래프를 실행 가능한 형태로 컴파일해 반환한다
    return graph.compile()


# ==================
# 15. 부팅 + 고정 진입점
# ==================

# 기본 소스로 구성한 전역 파이프라인 설정
PIPELINE_CONFIG = PipelineConfig(index=IndexConfig(sources=DEFAULT_SOURCES))

# 아래 5개는 bootstrap()이 채우기 전까지 None으로 시작하는 전역 상태값이다
_STORE: Optional[VectorStoreHandle] = None
_EMBEDDER: Optional[EmbedderHandle] = None
_GENERATOR: Optional[GeneratorHandle] = None
_GRAPH: Optional[Any] = None
INDEX_STATS: Optional[IndexStats] = None


def bootstrap(config: PipelineConfig = PIPELINE_CONFIG) -> None:
    # 함수 안에서 전역 변수를 재할당하기 위해 global로 선언한다
    global _STORE, _EMBEDDER, _GENERATOR, _GRAPH, INDEX_STATS

    if _GRAPH is not None:
        # 이미 초기화가 끝났으면(그래프가 존재하면) 아무 것도 하지 않고 즉시 반환한다 (멱등성)
        return

    # 인덱싱을 수행해 벡터 저장소, 임베더, 통계를 전역 변수에 채운다
    _STORE, _EMBEDDER, INDEX_STATS = build_index(config.index)
    # 인덱싱 통계를 로그로 남긴다
    print(f"[인덱싱 완료] {INDEX_STATS.model_dump()}")

    # 생성 모델을 로드해 전역 변수에 채운다
    _GENERATOR = load_generator(config.generation)
    # 노드들이 공유할 컨텍스트 객체를 만든다
    ctx = PipelineContext(
        store=_STORE, embedder=_EMBEDDER, generator=_GENERATOR, config=config
    )
    # 컨텍스트를 바탕으로 그래프를 빌드해 전역 변수에 채운다
    _GRAPH = build_graph(ctx)
    # 부팅이 끝났다는 로그를 남긴다
    print("[부팅 완료] run_rag_pipeline() 호출 준비됨")


def run_rag_pipeline(question: str) -> Dict[str, Any]:
    if not isinstance(question, str) or not question.strip():
        # 질문이 문자열이 아니거나 공백뿐이면 즉시 예외를 던진다
        raise ValueError("question은 비어 있지 않은 문자열이어야 합니다.")
    if _GRAPH is None:
        # 아직 bootstrap()이 호출되지 않았으면(그래프가 없으면) 예외를 던진다
        raise RuntimeError("bootstrap()이 완료되지 않았습니다.")

    # 그래프를 질문으로 초기화한 상태로 실행해 최종 상태를 얻는다
    final_state = _GRAPH.invoke(RagState(question=question.strip()))
    # 반환 타입이 dict인지 객체인지에 따라 payload를 꺼내는 방식을 분기한다
    payload = final_state["payload"] if isinstance(final_state, dict) else final_state.payload
    if isinstance(payload, dict):
        # payload가 dict 형태로 왔으면 AnswerPayload 객체로 다시 감싼다
        payload = AnswerPayload(**payload)
    # 결과기 반환 계약 형식(dict)으로 변환해 반환한다
    return payload.to_contract()


# =====================================================================================
# 16. 품질 결과 — 골드셋 채점
# =====================================================================================

def load_gold_set(path: str | Path) -> GoldSet:
    """gold_questions_public10.json → GoldSet."""
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    gold = GoldSet(questions=data["questions"], _meta=data.get("_meta", {}))
    print(f"[골드셋] {len(gold.questions)}문항 로드")
    return gold


def load_submission(path: str | Path) -> SubmissionFile:
    """answers_public_<팀>.json → SubmissionFile."""
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    sub = SubmissionFile(**data)
    print(f"[제출본] team={sub.team} · {len(sub.answers)}문항 로드")
    return sub


def score_article_exact_match(gold: GoldQuestion, answer: SubmissionAnswer) -> ArticleScore:
    """근거 조항 정확 일치 채점.

    문서명은 normalize_doc_name으로, 조번호는 "제3조"/"3"/3 모두 정수로 정규화해
    (문서명, 조번호) 집합으로 대조한다.
    hit_at_1은 예측 1순위가 정답 집합에 속하는지, hit_at_k는 정답 중 하나라도
    예측 안에 있는지를 뜻한다.
    """
    import re

    def to_key(document: Any, article: Any) -> Tuple[str, int]:
        digits = re.search(r"\d+", str(article))
        return normalize_doc_name(document), int(digits.group()) if digits else -1

    gold_keys = {to_key(item.doc, item.article) for item in gold.gold_articles}
    predicted_keys: List[Tuple[str, int]] = []
    for pair in answer.retrieved:
        if isinstance(pair, list) and len(pair) == 2:
            predicted_keys.append(to_key(pair[0], pair[1]))

    matched = gold_keys.intersection(predicted_keys)

    return ArticleScore(
        qid=gold.id,
        predicted=answer.retrieved,
        gold=[[item.doc, item.article] for item in gold.gold_articles],
        hit_at_1=bool(predicted_keys) and predicted_keys[0] in gold_keys,
        hit_at_k=bool(matched),
        n_gold_matched=len(matched),
        n_gold_total=len(gold_keys),
    )


def score_key_fact_coverage(gold: GoldQuestion, answer: SubmissionAnswer) -> KeyFactScore:
    """정답 핵심 사실 포함 비율.

    키팩트에서 숫자 토큰과 2자 이상 내용어 토큰을 뽑아 답변 안 존재 여부를 센다.
    숫자는 사실 판별에 결정적이므로 하나라도 빠지면 미포함으로 본다.
    자동 판정은 근사치이며 최종 확정은 수동 대조로 한다.
    """
    import re

    normalized_answer = re.sub(r"\s+", "", answer.answer)
    token_pattern = re.compile(r"[가-힣A-Za-z]{2,}|\d+")
    stopwords = {"있습니다", "합니다", "됩니다", "경우", "대하여", "또는", "그리고", "여러분"}

    per_fact: List[Dict[str, Any]] = []
    covered_count = 0

    for fact in gold.key_facts:
        tokens = token_pattern.findall(fact)
        numbers = [token for token in tokens if token.isdigit()]
        words = [
            token
            for token in tokens
            if not token.isdigit() and token not in stopwords
        ]
        words = sorted(set(words), key=len, reverse=True)[:8]

        numbers_hit = [number for number in numbers if number in normalized_answer]
        words_hit = [word for word in words if word in normalized_answer]

        numbers_ok = len(numbers_hit) == len(numbers)
        words_ratio = len(words_hit) / len(words) if words else 1.0
        covered = numbers_ok and words_ratio >= 0.6

        covered_count += int(covered)
        per_fact.append(
            {
                "fact": fact,
                "covered": covered,
                "numbers_expected": numbers,
                "numbers_hit": numbers_hit,
                "words_ratio": round(words_ratio, 3),
                "words_missing": [word for word in words if word not in words_hit],
            }
        )

    n_key_facts = len(gold.key_facts)
    return KeyFactScore(
        qid=gold.id,
        n_key_facts=n_key_facts,
        n_covered_auto=covered_count,
        coverage_auto=round(covered_count / n_key_facts, 4) if n_key_facts else 0.0,
        per_fact=per_fact,
        needs_manual_review=True,
    )


def evaluate(gold: GoldSet, submission: SubmissionFile) -> EvalReport:
    """GoldSet + SubmissionFile → EvalReport."""
    by_qid = {a.qid: a for a in submission.answers}
    items: List[ItemReport] = []

    for question in gold.questions:
        answer = by_qid.get(question.id)
        if answer is None:
            print(f"[경고] {question.id} 답변 누락 — 0점 처리")
            answer = SubmissionAnswer(qid=question.id, retrieved=[], answer="", error="missing")
        items.append(
            ItemReport(
                qid=question.id,
                question=question.question,
                difficulty=question.difficulty,
                ptype=question.ptype,
                article=score_article_exact_match(question, answer),
                key_fact=score_key_fact_coverage(question, answer),
                answer_text=answer.answer,
            )
        )

    n = max(1, len(items))
    return EvalReport(
        n_items=len(items),
        article_hit_at_1_rate=round(sum(i.article.hit_at_1 for i in items) / n, 4),
        article_hit_at_k_rate=round(sum(i.article.hit_at_k for i in items) / n, 4),
        key_fact_coverage_mean=round(sum(i.key_fact.coverage_auto for i in items) / n, 4),
        items=items,
    )


def render_manual_review(report: EvalReport) -> str:
    """EvalReport → 수동 대조용 마크다운 표.

    표 1은 문항별 점수 요약, 표 2는 미포함 판정된 키팩트와 생성 답변 대조용이다.
    """
    summary_lines = [
        "표 1. 문항별 자체 채점 요약",
        "| qid | 유형 | 난이도 | hit@1 | hit@k | 조항 일치 | 키팩트 커버리지 |",
        "|---|---|---|---|---|---|---|",
    ]
    for item in report.items:
        summary_lines.append(
            f"| {item.qid} | {item.ptype} | {item.difficulty} | "
            f"{'O' if item.article.hit_at_1 else 'X'} | "
            f"{'O' if item.article.hit_at_k else 'X'} | "
            f"{item.article.n_gold_matched}/{item.article.n_gold_total} | "
            f"{item.key_fact.n_covered_auto}/{item.key_fact.n_key_facts} |"
        )

    summary_lines.append("")
    summary_lines.append(
        f"종합: hit@1 {report.article_hit_at_1_rate} · hit@k {report.article_hit_at_k_rate} "
        f"· 키팩트 커버리지 평균 {report.key_fact_coverage_mean} · {report.n_items}문항"
    )

    detail_lines = ["", "표 2. 수동 대조 필요 항목 (자동 판정 미포함)",
                    "| qid | 누락 키팩트 | 누락 토큰 |", "|---|---|---|"]
    for item in report.items:
        for fact_detail in item.key_fact.per_fact:
            if fact_detail.get("covered"):
                continue
            fact_text = str(fact_detail["fact"])[:60]
            missing = ", ".join(fact_detail.get("words_missing", [])[:5])
            detail_lines.append(f"| {item.qid} | {fact_text} | {missing} |")

    return "\n".join(summary_lines + detail_lines)


# =====================================================================================
# 17. 품질 결과 — 성능 사전 검증
# =====================================================================================

def measure_performance(protocol: PerfProtocol = PerfProtocol()) -> PerfReport:
    """공식 프로토콜과 동일 조건으로 p50/p95/throughput 자체 측정.

    공통 러너와 같은 closed-loop 방식으로 로컬 HTTP 서버에 동시 요청을 보내고,
    반복 측정 결과의 중앙값을 대표값으로 삼는다.
    """
    import statistics
    from concurrent.futures import ThreadPoolExecutor

    import requests

    base_url = "http://127.0.0.1:8000"
    sample_question = "사업자/단체 카카오계정은 담당자 몇 명이 이용할 수 있나요?"

    def send_one(_: int) -> Dict[str, Any]:
        started = time.perf_counter()
        try:
            response = requests.post(
                f"{base_url}/answer", json={"question": sample_question}, timeout=120
            )
            response.raise_for_status()
            return {"ok": True, "latency_s": time.perf_counter() - started}
        except Exception as exc:  # noqa: BLE001
            return {
                "ok": False,
                "latency_s": time.perf_counter() - started,
                "error": f"{type(exc).__name__}: {exc}",
            }

    def run_round(n_requests: int) -> Dict[str, Any]:
        started = time.perf_counter()
        with ThreadPoolExecutor(max_workers=max(1, protocol.concurrency)) as pool:
            rows = list(pool.map(send_one, range(n_requests)))
        wall_s = time.perf_counter() - started
        ok_rows = [row for row in rows if row["ok"]]
        latencies = sorted(row["latency_s"] for row in ok_rows)
        return {
            "success": len(ok_rows),
            "total": len(rows),
            "wall_s": wall_s,
            "throughput_rps": len(ok_rows) / wall_s if wall_s else 0.0,
            "latencies": latencies,
        }

    def percentile(values: List[float], ratio: float) -> Optional[float]:
        if not values:
            return None
        position = min(len(values) - 1, max(0, int(round(ratio * (len(values) - 1)))))
        return round(values[position], 4)

    if protocol.warmup_requests > 0:
        print(f"[성능] 워밍업 {protocol.warmup_requests}요청")
        run_round(protocol.warmup_requests)

    samples: List[Dict[str, Any]] = []
    for repetition in range(1, protocol.repetitions + 1):
        print(f"[성능] 측정 {repetition}/{protocol.repetitions}")
        samples.append(run_round(protocol.requests_per_run))

    success_rates = [row["success"] / row["total"] for row in samples]
    throughputs = [row["throughput_rps"] for row in samples]
    p50_values = [
        value for value in (percentile(row["latencies"], 0.50) for row in samples)
        if value is not None
    ]
    p95_values = [
        value for value in (percentile(row["latencies"], 0.95) for row in samples)
        if value is not None
    ]

    report = PerfReport(
        protocol=protocol,
        success_rate=round(statistics.median(success_rates), 4),
        throughput_rps=round(statistics.median(throughputs), 4),
        p50_latency_s=round(statistics.median(p50_values), 4) if p50_values else None,
        p95_latency_s=round(statistics.median(p95_values), 4) if p95_values else None,
    )
    print(
        f"[성능] 성공률 {report.success_rate} · {report.throughput_rps} req/s "
        f"· p50 {report.p50_latency_s}s · p95 {report.p95_latency_s}s"
    )
    return report


# =====================================================================================
# 18. 고정 FastAPI 연결 영역
# =====================================================================================

def create_app():
    """전역 FastAPI app 생성. GET /health, POST /answer 고정."""
    from fastapi import FastAPI, HTTPException

    fastapi_app = FastAPI(title="KTB AI Performance Result Generator")
    lock = threading.Lock()

    @fastapi_app.get("/health")
    def health() -> Dict[str, str]:
        return {"status": "ok"}

    @fastapi_app.post("/answer")
    def answer_api(payload: dict) -> Dict[str, Any]:
        question = payload.get("question")
        if not isinstance(question, str) or not question.strip():
            raise HTTPException(status_code=400, detail="question must be a non-empty string")
        with lock:
            return run_rag_pipeline(question.strip())

    return fastapi_app


try:
    app = create_app()
except ImportError:
    app = None
    print("[주의] fastapi 미설치 — 스켈레톤 검증 모드로 app=None")


# =====================================================================================
# 19. 스켈레톤 흐름 확인
# =====================================================================================

def smoke_test(gold_path: Optional[str] = None, answers_path: Optional[str] = None) -> None:
    """인덱싱 → 검색 → 증강 → 생성 → 품질 결과까지 한 번 흘려본다."""
    print("=" * 70)
    bootstrap()

    print("=" * 70)
    result = run_rag_pipeline("사업자/단체 카카오계정은 담당자 몇 명이 이용할 수 있나요?")
    print(f"[run_rag_pipeline 반환] {json.dumps(result, ensure_ascii=False)}")

    if gold_path and answers_path:
        print("=" * 70)
        report = evaluate(load_gold_set(gold_path), load_submission(answers_path))
        print(render_manual_review(report))

    print("=" * 70)
    print(f"[완료] 스텁 잔여 호출 {len(STUB_LOG)}회")

# 스켈레톤 흐름 확인 — 골드셋/제출본 경로가 있으면 함께 인자로 전달
smoke_test()
